In [1]:
import cv2
import numpy as np
from ultralytics import YOLO

In [2]:
# 1. Load your best model
model = YOLO('../best_yolov8_coral_reef/runs/detect/reef_coral/weights/best.pt')

# 2. Setup Video
video_path = '../yolov8_model/videos/Match 6 (R2) - 2025 Central Missouri Regional.mp4'
cap = cv2.VideoCapture(video_path)

In [3]:

OUTPUT_PATH = 'debug_scoring_custom_caps.mp4'

# NEW: Specific capacity for each of the 6 slots (Left to Right)
SLOT_CAPACITIES = [3, 1, 2, 2, 1, 3]
NUM_SLOTS = len(SLOT_CAPACITIES)
# --------------

# Video properties and timing
width  = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
fps    = cap.get(cv2.CAP_PROP_FPS)
total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
cutoff_frame = total_frames - int(49 * fps)

# Crop setup (Bottom 2/5ths)
crop_h = int(height * (2/5))
start_y = height - crop_h

# Video Writer
fourcc = cv2.VideoWriter_fourcc(*'mp4v')
out = cv2.VideoWriter(OUTPUT_PATH, fourcc, fps, (width, crop_h))

# Tracking state
reef_grids = [[0] * NUM_SLOTS for _ in range(5)]
claimed_coral_ids = set()
total_points = 0
current_frame_idx = 0

# --- NEW CONFIG & LOGGING ---
STABILITY_THRESHOLD = 10 
potential_scores = {}    
scoring_log = [] # To store (timestamp, reef_id, slot_idx)
# ----------------------------

while cap.isOpened():
    ret, frame = cap.read()
    if not ret or current_frame_idx >= cutoff_frame:
        break

    cropped = frame[start_y:height, 0:width]
    results = model.track(cropped, persist=True, tracker="botsort.yaml", conf=0.25, verbose=False)
    annotated_frame = results[0].plot()

    if results[0].boxes.id is not None:
        boxes = results[0].boxes.xyxy.cpu().numpy()
        clss = results[0].boxes.cls.cpu().numpy().astype(int)
        ids = results[0].boxes.id.cpu().numpy().astype(int)

        frame_reefs = [boxes[i] for i, c in enumerate(clss) if model.names[c] == 'reef']
        frame_reefs.sort(key=lambda x: x[0])

        current_frame_coral_ids = set()

        for i, c_id in enumerate(ids):
            if model.names[clss[i]] == 'coral':
                if c_id in claimed_coral_ids:
                    continue
                
                current_frame_coral_ids.add(c_id)
                cx, cy = (boxes[i][0] + boxes[i][2])/2, (boxes[i][1] + boxes[i][3])/2
                
                in_scoring_zone = False
                for r_idx, r_box in enumerate(frame_reefs):
                    rx1, ry1, rx2, ry2 = r_box
                    rw, rh = rx2 - rx1, ry2 - ry1
                    
                    if (rx1 < cx < rx2) and (ry1 < cy < ry1 + (rh * 0.20)):
                        in_scoring_zone = True
                        rel_x = cx - rx1
                        slot_idx = int((rel_x / rw) * NUM_SLOTS)
                        slot_idx = max(0, min(slot_idx, NUM_SLOTS - 1))

                        if c_id in potential_scores:
                            p = potential_scores[c_id]
                            if p["reef"] == r_idx and p["slot"] == slot_idx:
                                p["count"] += 1
                            else:
                                potential_scores[c_id] = {"reef": r_idx, "slot": slot_idx, "count": 1}
                        else:
                            potential_scores[c_id] = {"reef": r_idx, "slot": slot_idx, "count": 1}

                        if potential_scores[c_id]["count"] >= STABILITY_THRESHOLD:
                            max_allowed = SLOT_CAPACITIES[slot_idx]
                            if r_idx < len(reef_grids) and reef_grids[r_idx][slot_idx] < max_allowed:
                                # --- CALC TIMESTAMP ---
                                seconds = current_frame_idx / fps
                                timestamp = f"{int(seconds // 60):02d}:{int(seconds % 60):02d}"
                                
                                reef_grids[r_idx][slot_idx] += 1
                                claimed_coral_ids.add(c_id)
                                total_points += 4
                                
                                # Log the event
                                scoring_log.append((timestamp, r_idx, slot_idx))
                                print(f"[{timestamp}] SCORE! Reef {r_idx}, Slot {slot_idx}. Total: {total_points}")
                                
                                del potential_scores[c_id]
                                cv2.circle(annotated_frame, (int(cx), int(cy)), 25, (0, 255, 0), 3)
                        break 
                
                if not in_scoring_zone and c_id in potential_scores:
                    del potential_scores[c_id]

        for pid in list(potential_scores.keys()):
            if pid not in current_frame_coral_ids:
                del potential_scores[pid]

    out.write(annotated_frame)
    current_frame_idx += 1

cap.release()
out.release()

# --- FINAL SUMMARY WITH LOG ---
print("\n" + "="*45)
print("SCORING TIMELINE")
print("-" * 45)
for entry in scoring_log:
    print(f"Time: {entry[0]} | Reef: {entry[1]} | Slot: {entry[2]}")

# (Keep your previous Reef Grid table printout here as well)

[00:12] SCORE! Reef 1, Slot 5. Total: 4
[00:14] SCORE! Reef 1, Slot 0. Total: 8
[00:25] SCORE! Reef 1, Slot 5. Total: 12
[00:33] SCORE! Reef 0, Slot 2. Total: 16
[00:35] SCORE! Reef 1, Slot 2. Total: 20
[00:39] SCORE! Reef 0, Slot 3. Total: 24
[00:40] SCORE! Reef 0, Slot 3. Total: 28
[00:41] SCORE! Reef 1, Slot 2. Total: 32
[00:41] SCORE! Reef 1, Slot 3. Total: 36
[00:42] SCORE! Reef 1, Slot 1. Total: 40
[00:43] SCORE! Reef 1, Slot 0. Total: 44
[00:43] SCORE! Reef 1, Slot 0. Total: 48
[00:46] SCORE! Reef 1, Slot 3. Total: 52
[00:48] SCORE! Reef 0, Slot 0. Total: 56
[00:54] SCORE! Reef 1, Slot 4. Total: 60
[00:55] SCORE! Reef 1, Slot 5. Total: 64
[00:58] SCORE! Reef 0, Slot 1. Total: 68
[01:11] SCORE! Reef 0, Slot 0. Total: 72
[01:12] SCORE! Reef 0, Slot 0. Total: 76
[01:38] SCORE! Reef 0, Slot 4. Total: 80
[01:48] SCORE! Reef 0, Slot 5. Total: 84
[02:10] SCORE! Reef 0, Slot 2. Total: 88
[02:20] SCORE! Reef 0, Slot 5. Total: 92

SCORING TIMELINE
-----------------------------------------

In [ ]:
print("\n" + "="*30)
print("FINAL REEF OCCUPANCY REPORT")
print("="*30)

for r_idx, slots in enumerate(reef_grids):
    # Only print reefs that actually had at least one coral detected
    if sum(slots) > 0:
        # Create a visual string: [0, 2, 1, 0, 0, 0] -> "Slot 0: 0 | Slot 1: 2 | Slot 2: 1 ..."
        status_str = " | ".join([f"Slot {i}: {val}" for i, val in enumerate(slots)])
        print(f"REEF {r_idx}: {status_str}")
        
        # Optional: Print total corals on this specific reef
        print(f"   -> Total Corals on Reef {r_idx}: {sum(slots)}")
        print("-" * 30)

print(f"GRAND TOTAL SCORE: {total_points} points")
print("="*30)

Starting analysis with custom slot caps: [3, 1, 2, 2, 1, 3]

SECTION    | SLOT 0 SLOT 1 SLOT 2 SLOT 3 SLOT 4 SLOT 5
CAPACITY   | (3)    (1)    (2)    (2)    (1)    (3)   
---------------------------------------------
GRAND TOTAL SCORE: 0 points
